<a href="https://colab.research.google.com/github/Chris-p05/NPL-Project/blob/main/OptiHireDataScraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [88]:
# --- STEP 1: GLOBAL SETUP & LIBRARIES ---
!pip install torch tensorflow transformers nltk spacy pymupdf scrapingbee kaggle -q
import os, json, base64, requests, pymupdf, spacy, pandas as pd
from google.colab import userdata

In [89]:
# --- STEP 2: SECURE CREDENTIALS --- (SET THEM UP ON THE SECRETS TAB)
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
# --- STEP 3: DATA ACQUISITION (THE "GOOD" DATASET) ---
!kaggle datasets download -d snehaanbhawal/resume-dataset
!unzip -o resume-dataset.zip

In [91]:
# --- STEP 4: PROFESSIONAL LOGIC ENGINE ---
try:
    nlp = spacy.load("en_core_web_md")
except OSError:
    spacy.cli.download("en_core_web_md")
    nlp = spacy.load("en_core_web_md")

def clean_and_anonymize(text):
    """
    Combines PII redaction and text normalization into one high-tier pass.

    """
    if not isinstance(text, str): return ""

    # 1. Anonymize to eliminate human bias
    doc = nlp(text)
    redacted_text = text
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "GPE", "DATE"]:
            redacted_text = redacted_text.replace(ent.text, f"[{ent.label_}_REDACTED]")

    # 2. Normalize for Semantic Analysis
    clean_doc = nlp(redacted_text.lower())
    tokens = [t.lemma_ for t in clean_doc if not t.is_stop and not t.is_punct]
    return " ".join(tokens)

def get_base64_resume(file_path):
    with open(file_path, "rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')

In [ ]:
# --- STEP 5: BATCH PROCESSING (KAGGLE DATA) --- THIS IS WHERE WE TEST A SPECIFIC RESUME AGAINST THE DATASET
# We use 'Resume.csv' and 'Resume_str' from the new dataset.
try:
    df = pd.read_csv('Resume/Resume.csv') # Adjust path if unzip structure differs
except:
    df = pd.read_csv('Resume.csv')

processed_data = []
print("Processing the REAL resume text now!")

# Processing 50 samples to test the logic
for index, row in df.head(50).iterrows():
    processed_data.append({
        "id": row['ID'],
        "category": row['Category'],
        "clean_resume": clean_and_anonymize(row['Resume_str'])
    })

In [ ]:
# --- STEP 6: SCRAPINGBEE & INDIVIDUAL EVALUATION --- (AGAIN, SET THEM UP IN SECRETS TAB)
# Connects your live job data to a specific file upload.
try:
    client = ScrapingBeeClient(api_key=userdata.get('SCRAPINGBEE_KEY'))

    response = client.get(
        'https://www.simplyhired.com/search?q=python+developer',
        params={'ai_query': 'Extract job title, company, requirements, and responsibilities', 'render_js': True}
    )
    job_data = response.json()
except Exception as e:
    print(f"ScrapingBee Error: {e}. Check your API key!")

# Evaluate "my_resume.pdf" if the user managed to upload it.
pdf_file = "my_resume.pdf"
if os.path.exists(pdf_file):
    encoded_resume = get_base64_resume(pdf_file)
    print(f"Ready to evaluate {pdf_file} against {job_data.get('job_title', 'Job Posting')}")
else:
    print(f"Warning: {pdf_file} not found. Upload it to the folder!")

print(f"\nPhase 1 & 2 COMPLETE. {len(processed_data)} resumes processed with zero bias.")